# Bottleneck estimates for the murine pneumonia model
Uses `bottleneck_from_two_timepoints` from the supplied Python module without changing the estimator. For each time point, noDox uses its **Pre1** and Dox uses its **Pre2** donor.

Keep the raw-count CSV and Python module beside this notebook. Requires NumPy, pandas, SciPy, and g++ or clang++ with C++17 support. The module compiles its C++ code automatically.

The updated Table S7 includes nine 48 hpi Dox lung samples (mice 1, 4, 5, 6, 8, 9, 10, 12, and 13), aligned by sgRNA ID from the raw 48-hour workbook. These use the 48 hpi Pre2 donor. Select samples by their experiment header; exclude the influenza superinfection sections. There are 65 recipient samples in total and no 24 hpi blood samples.


In [ ]:
from pathlib import Path
import importlib.util
import re
import numpy as np
import pandas as pd

DATA_FILE = Path("Table-S7-sgRNA-Raw-Counts.csv")
FUNCTION_FILE = Path("bottleneck_function.py")
# Support the filename of the attached module as well.
if not FUNCTION_FILE.exists():
    FUNCTION_FILE = Path("bottleneck_function(3).py")
OUTPUT_FILE = Path("bottlenecks_24_48hpi.csv")
NB_MAX = 5_000_000
DONOR_BY_CONDITION = {"noDox": "Pre1", "Dox": "Pre2"}

spec = importlib.util.spec_from_file_location("bottleneck_function", FUNCTION_FILE)
bf = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bf)

In [ ]:
# Read both header rows explicitly: sample names repeat across experiments.
raw = pd.read_csv(DATA_FILE, header=None, dtype=str)
section = raw.iloc[0].fillna("").str.strip()
section = section.mask(section.str.startswith("Unnamed:") | section.eq("")).ffill()
sample_names = raw.iloc[1].fillna("").str.strip()
counts = raw.iloc[2:].loc[raw.iloc[2:, 0].str.fullmatch(r"sgRNA\d+", na=False)].reset_index(drop=True)

metadata = []
donor_columns = {}
for hpi in (24, 48):
    donor_section = f"preinfection samples for {hpi} hpi of murine pneumonia model"
    recipient_section = f"{hpi} hpi of murine pneumonia model"
    for donor in ("Pre1", "Pre2"):
        matches = [j for j in range(raw.shape[1])
                   if section.iloc[j] == donor_section and sample_names.iloc[j] == donor]
        if len(matches) != 1:
            raise ValueError(f"Expected one {hpi} hpi {donor} column, found {matches}")
        donor_columns[(hpi, donor)] = matches[0]
    for j in range(raw.shape[1]):
        if section.iloc[j] != recipient_section:
            continue
        sample = sample_names.iloc[j]
        if hpi == 24:
            match = re.fullmatch(r"Mice_(noDox|plusDox)_(\d+)", sample)
            if match is None:
                raise ValueError(f"Unrecognized 24 hpi sample: {sample}")
            group, mouse = match.groups()
            tissue = "Lung"
        else:
            match = re.fullmatch(r"Mice_(noDox|Dox)-(\d+)-(Lung|Blood)", sample)
            if match is None:
                raise ValueError(f"Unrecognized 48 hpi sample: {sample}")
            group, mouse, tissue = match.groups()
        condition = "noDox" if group == "noDox" else "Dox"
        metadata.append(dict(hpi=hpi, condition=condition, mouse=int(mouse),
                             tissue=tissue, sample=sample,
                             donor_sample=DONOR_BY_CONDITION[condition], column=j))

samples = pd.DataFrame(metadata)
if samples.duplicated(["hpi", "condition", "mouse", "tissue"]).any():
    raise ValueError("Duplicate recipient identities")

# Verify that the updated Dox lung data were selected.
dox_lung = samples.query("hpi == 48 and condition == 'Dox' and tissue == 'Lung'")
if sorted(dox_lung["mouse"].tolist()) != [1, 4, 5, 6, 8, 9, 10, 12, 13]:
    raise ValueError("Expected the nine Dox lung samples from updated Table S7")

def count_vector(column):
    x = pd.to_numeric(counts.iloc[:, column], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(x).all() or (x < 0).any() or (x != np.floor(x)).any():
        raise ValueError(f"Invalid or missing counts in column {column}")
    if x.sum() <= 0:
        raise ValueError(f"Empty sample in column {column}")
    return x.astype(np.int64)

# Validate all selected counts before starting estimation. Row order is preserved.
vectors = {j: count_vector(j) for j in
           set(donor_columns.values()) | set(samples["column"])}
display(samples.groupby(["hpi", "condition", "tissue"]).size().rename("n_samples"))


In [ ]:
results = []
for i, sample in enumerate(samples.to_dict("records"), start=1):
    donor = vectors[donor_columns[(sample["hpi"], sample["donor_sample"])]]
    recipient = vectors[sample["column"]]
    row = {key: value for key, value in sample.items() if key != "column"}
    row.update(read_depth=int(recipient.sum()),
               assumption="global_neutrality_q_equals_D",
               richness=int(np.count_nonzero(recipient)))
    try:
        estimate = bf.bottleneck_from_two_timepoints(
            donor, recipient, donor_pseudocount=1e-6, nb_max=NB_MAX)
        row.update(B_hat=estimate,
                   status="infinite_boundary" if np.isinf(estimate) else "finite",
                   error="")
    except Exception as exc:
        # Keep failed samples visible rather than silently dropping them.
        row.update(B_hat=np.nan, status="error", error=f"{type(exc).__name__}: {exc}")
    results.append(row)
    print(f"[{i}/{len(samples)}] {sample['hpi']} hpi {sample['sample']}: "
          f"B_hat={row['B_hat']}, status={row['status']}", flush=True)

bottlenecks = pd.DataFrame(results)
bottlenecks.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(bottlenecks)} rows to {OUTPUT_FILE.resolve()}")
display(bottlenecks)
if bottlenecks["status"].eq("error").any():
    print("Some estimates failed. Inspect the error column before using these results.")